# Math Quiz and Analysis

This notebook processes math assessment data, creates a SQLite database, and generates visualizations.



In [2]:
## Setup and Imports

import os
import random
import json
import uuid
import time
#import tkinter as tk
#from tkinter import ttk
# import pandas as pd
# import sqlite3
# import seaborn as sns
# import matplotlib.pyplot as plt
# import numpy as np
from datetime import datetime

In [3]:
## USER AND SETTINGS

# Default Settings
settings = {
    "account_name": "test",
    "num_problems": 5,
    "number_range": (0, 5),  # tuple of (min, max) range of numbers to use, inclusive
    "numbers_include": [],  # list of numbers to include
    "numbers_exclude": [],  # list of numbers to exclude
    "num_numbers": 2,
    "operations": ['+']
}

# Real Settings
# settings["account_name"] = "Randy"
# settings["num_problems"] = 20
# settings["number_range"] = [0, 9]
# settings["numbers_include"] = []
# settings["numbers_exclude"] = []
# settings["num_numbers"] = 2
# settings["operations"] = ['+']

# Print the current settings
print("Current settings:")
for key, value in settings.items():
    print(f"{key}: {value}")


Current settings:
account_name: test
num_problems: 5
number_range: (0, 5)
numbers_include: []
numbers_exclude: []
num_numbers: 2
operations: ['+']


In [ ]:
### QUIZ
def generate_problem(number_range, operations=["+"], number_include=None, number_exclude=None, num_numbers=2):
    if (not number_range or number_range == [0, 0]) and number_include:
        available_numbers = number_include
    else:
        available_numbers = [num for num in number_range if num not in (number_exclude or [])]
        if number_include:
            available_numbers.extend(number_include)
    
    if not available_numbers:
        raise ValueError("No available numbers to generate a problem.")
    
    numbers = [random.choice(available_numbers) for _ in range(num_numbers)]
    
    operation = random.choice(operations)
    problem_string = f"{numbers[0]} {operation} {numbers[1]}"
    
    # Replace operation symbols for display
    display_problem = problem_string.replace("**", "^").replace("*", "×").replace("/", "÷")
    
    # Calculate correct answer based on the operation
    if operation == "+":
        correct_answer = sum(numbers)
    elif operation == "-":
        correct_answer = numbers[0] - numbers[1]
    elif operation == "*":
        correct_answer = numbers[0] * numbers[1]
    elif operation == "/":
        correct_answer = numbers[0] / numbers[1]
    elif operation == "^":
        correct_answer = numbers[0] ** numbers[1]
    else:
        raise ValueError(f"Unsupported operation: {operation}")
    
    return display_problem, correct_answer

def assign_problem_id(problem_string):
    parts = problem_string.split()
    
    if len(parts) == 3:  # Two numbers and one operation
        num1, operation, num2 = parts
        op_codes = {
            '+': 'add',
            '-': 'sub',
            '×': 'mul',
            '÷': 'div',
            '^': 'exp'
        }
        op_code = op_codes.get(operation, 'unknown')
        return f"{op_code}_{num1}_{num2}"
    else:
        # Create a unique ID for problems with more than two numbers or multiple operations
        unique_id = uuid.uuid4().hex[:8]  # Use first 8 characters of a UUID
        return f"prob_{unique_id}"

def write_session_json(session_data, folder_path='apps/math-quiz/math-quiz_data'):
    filename = f"math_session_{session_data['user']['name']}_{session_data['session']['start_time']}.json"
    filepath = os.path.join(folder_path, filename)
    os.makedirs(folder_path, exist_ok=True)
    with open(filepath, 'w') as json_file:
        json.dump(session_data, json_file, indent=2)
    print(f"\nSession data saved to {filepath}")

def display_summary(session_summary, incorrect_problems):
    print("Session Summary:")
    print(f"Total problems attempted: {session_summary['total_problems']}")
    print(f"Number of correct answers: {session_summary['correct_answers']}")
    print(f"Average response time: {session_summary['average_response_time_seconds']:.1f} seconds")
    if incorrect_problems:
        print("Incorrectly answered problems:")
        for p in incorrect_problems:
            print(f"{p['question']} (Your answer: {p['user_answer']}, Correct answer: {p['correct_answer']})")
    else:
        print("All answers were correct!")

from ipywidgets import widgets
from IPython.display import display, clear_output

def run_assessment(settings):
    from primary.fileops import get_current_datetime_filefriendly
    from ipywidgets import widgets
    from IPython.display import display, clear_output
    
    # Initialize session data
    session_id = str(uuid.uuid4())
    session_start_time = get_current_datetime_filefriendly()
    problems_attempted = []
    used_problems = set()

    print("\nStarting the assessment...\n")

    for _ in range(settings['num_problems']):
        # Generate a unique problem
        while True:
            display_problem, correct_answer = generate_problem(
                settings['number_range'],
                operations=settings['operations'],
                numbers_include=settings.get('numbers_include'),
                numbers_exclude=settings.get('numbers_exclude'),
                num_numbers=settings['num_numbers']
            )
            problem_id = assign_problem_id(display_problem)
            if problem_id not in used_problems:
                used_problems.add(problem_id)
                break

        print(f"Solve: {display_problem}")
        
        # Create input widget
        answer_input = widgets.Text(description="Your answer:", continuous_update=False)
        output = widgets.Output()
        
        # Display widgets
        display(answer_input, output)
        
        # Initialize variables
        answer_submitted = False
        start_time = time.time()
        
        def on_value_change(change):
            nonlocal answer_submitted
            if change['type'] == 'change' and change['name'] == 'value':
                if not answer_submitted:
                    with output:
                        clear_output()
                        end_time = time.time()
                        user_answer_string = change['new']
                        
                        # Input validation
                        try:
                            user_answer_numeric = float(user_answer_string)
                            user_answer_int = int(user_answer_numeric)
                            is_correct = abs(user_answer_numeric - correct_answer) < 1e-6
                        except ValueError:
                            print("Invalid input. Please enter a number.")
                            is_correct = False
                            user_answer_numeric = None
                            user_answer_int = None

                        response_time_ms = round((end_time - start_time) * 1000)

                        if is_correct:
                            print("Correct!\n")
                        else:
                            print(f"Incorrect. The correct answer was {correct_answer}.\n")

                        # Record problem data
                        problem_data = {
                            "id": problem_id,
                            "question": display_problem,
                            "correct_answer": correct_answer,
                            "user_answer_string": user_answer_string,
                            "user_answer": user_answer_int,
                            "is_correct": is_correct,
                            "response_time_ms": response_time_ms
                        }
                        problems_attempted.append(problem_data)
                    
                    answer_submitted = True
        
        # Observe the value change of the input widget
        answer_input.observe(on_value_change, names='value')
        
        # Wait for user input
        while not answer_submitted:
            time.sleep(0.1)

    # Session end
    session_end_time = get_current_datetime_filefriendly()
    total_problems = len(problems_attempted)
    correct_answers = sum(1 for p in problems_attempted if p["is_correct"])
    average_response_time_ms = sum(p["response_time_ms"] for p in problems_attempted) / total_problems
    incorrect_problems = [p for p in problems_attempted if not p["is_correct"]]

    # Prepare session summary and display it
    session_summary = {
        "total_problems": total_problems,
        "correct_answers": correct_answers,
        "average_response_time_seconds": average_response_time_ms / 1000
    }
    display_summary(session_summary, incorrect_problems)

    # Prepare JSON data
    session_data = {
        "version": "1.0",
        "timestamp": datetime.utcnow().isoformat(),
        "user": {
            "id": str(uuid.uuid4()),
            "name": settings['account_name']
        },
        "session": {
            "id": session_id,
            "start_time": session_start_time,
            "end_time": session_end_time,
            "settings": {
                "operations": settings['operations'],
                "number_range": list(settings['number_range']),
                "numbers_include": settings.get('numbers_include'),
                "numbers_exclude": settings.get('numbers_exclude'),
                "num_numbers": settings['num_numbers']
            },
            "problems": problems_attempted,
            "summary": {
                "total_problems": total_problems,
                "correct_answers": correct_answers,
                "average_response_time_ms": round(average_response_time_ms)
            }
        }
    }

    # Write JSON data to file
    write_session_json(session_data)

# Run the assessment
settings = {
    'account_name': 'test_user',
    'num_problems': 5,
    'number_range': range(1, 11),
    'num_numbers': 2,
    'operations': ['+'],
    'numbers_include': None,
    'numbers_exclude': None
}
run_assessment(settings)

In [ ]:
## Setup and Imports

import os
import random
import json
import uuid
import time
#import tkinter as tk
#from tkinter import ttk
# import pandas as pd
# import sqlite3
# import seaborn as sns
# import matplotlib.pyplot as plt
# import numpy as np
from datetime import datetime

In [ ]:
### MANUAL TESTS

def manual_tests():
     # Test generate_problem function
    print("Testing generate_problem function:")
    number_range = list(range(1, 11))
    operations = ["+"]
    
    print("Test 1: Basic addition")
    problem, answer = generate_problem(number_range, operations)
    print(f"Problem: {problem}, Answer: {answer}")
    
    print("\nTest 2: With number_include")
    problem, answer = generate_problem(number_range, operations, number_include=[15, 20])
    print(f"Problem: {problem}, Answer: {answer}")
    
    print("\nTest 3: With number_exclude")
    problem, answer = generate_problem(number_range, operations, number_exclude=[1, 2, 3])
    print(f"Problem: {problem}, Answer: {answer}")
    
    # Test assign_problem_id function
    print("\nTesting assign_problem_id function:")
    
    print("Test 1: Simple addition")
    problem_id = assign_problem_id("5 + 3")
    print(f"Problem: 5 + 3, ID: {problem_id}")
    
    print("\nTest 2: Multiplication")
    problem_id = assign_problem_id("4 × 6")
    print(f"Problem: 4 × 6, ID: {problem_id}")
    
    print("\nTest 3: Complex problem")
    problem_id = assign_problem_id("2 + 3 + 4")
    print(f"Problem: 2 + 3 + 4, ID: {problem_id}")

manual_tests()